In [ ]:
from river import compose, datasets, metrics, preprocessing, anomaly
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from numpy import linspace

In [ ]:
dataset = datasets.CreditCard().take(10000)

model = compose.Pipeline(
    preprocessing.MinMaxScaler(),
    anomaly.HalfSpaceTrees()
)

auc = metrics.ROCAUC()

for x, y in dataset:
    score = model.score_one(x)
    model.learn_one(x)
    auc.update(y, score)
    
auc

ROCAUC: 94.38%

## Thresholding Strategy and Evaluation

We will use the following approach:


- Collect anomaly scores and true labels for a batch of samples.
- Choose the threshold that maximizes the F1-score on this batch.
- Use this threshold to convert scores to binary predictions.
- Evaluate the predictions using confusion matrix, precision, recall, and F1-score.


In [13]:
scores = []
labels = []
for x, y in datasets.CreditCard().take(10000):
    score = model.score_one(x)
    scores.append(score)
    labels.append(y)
    model.learn_one(x)

# Find the best threshold for F1-score
best_f1 = 0
best_thresh = 0
for thresh in linspace(min(scores), max(scores), 100):
    preds = [int(s > thresh) for s in scores]
    f1 = f1_score(labels, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"Best threshold: {best_thresh:.4f}, Best F1: {best_f1:.4f}")

Best threshold: 0.8306, Best F1: 0.5763


In [14]:
# Use the best threshold to make predictions
preds = [int(s > best_thresh) for s in scores]

# Evaluate
cm = confusion_matrix(labels, preds)
precision = precision_score(labels, preds)
recall = recall_score(labels, preds)
f1 = f1_score(labels, preds)

print("Confusion Matrix:\n", cm)
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

Confusion Matrix:
 [[9958    4]
 [  21   17]]
Precision: 0.8095
Recall: 0.4474
F1-score: 0.5763
